# Attention Weight Visualization

Visualize envelope + attention weights from both the Rhythm Regressor and Duration Regressor models.

**Usage:** Modify `UTT_ID` and `SPK_ID` in the last cell, then re-run it to explore different utterances.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import Audio, display, Markdown
from pathlib import Path
import sys
import os
#sys.path.insert(0, str(Path.cwd().resolve().parent))
os.chdir(str(Path.cwd().resolve().parent))

from src.visualization_utils import *

# --- Configuration ---
RHYTHM_CONFIG = 'config/exp_A3c_envRate_prosody_expand.json'
RHYTHM_CKPT = 'exp/exp_A3c_envRate_prosody_expand/checkpoints/20260614_1649/best_model_epoch114.pth'
DURATION_CONFIG = 'config/exp_A2c_durZ_prosody.json'
DURATION_CKPT = 'exp/exp_A2c_durZ_prosody/checkpoints/20260613_1333/best_model_epoch66.pth'
LMDB_PATH = 'data/speechocean/rtm_feats.lmdb'
VC_FEATURES_PATH = 'data/speechocean/vc_features.json'
VC_ALIGNMENTS_PATH = 'data/speechocean/vc_alignments.json'
METADATA_PATH = 'data/speechocean/speechocean_metadata.csv'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

/home/joao.lima/miniconda3/envs/rtm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
# Load models (run once)
rhythm_model, rhythm_config = load_model(RHYTHM_CONFIG, RHYTHM_CKPT, DEVICE)
print(f"Rhythm model loaded: {rhythm_config['exp_name']}")
rmp = rhythm_config['model_params']
ds_factor = get_envelope_downsample_factor(rmp)
print(f"  CNN stride={rmp.get('cnn_stride')}, layers={rmp.get('num_cnn_layers')} -> {ds_factor}x downsampling")

duration_model, duration_config = load_model(DURATION_CONFIG, DURATION_CKPT, DEVICE)
print(f"Duration model loaded: {duration_config['exp_name']}")

transcriptions = load_transcriptions()
print(f"Loaded {len(transcriptions)} transcriptions")

Rhythm model loaded: exp_A3c_envRate_prosody_expand
  CNN stride=3, layers=2 -> 9x downsampling
Duration model loaded: exp_A2c_durZ_prosody
Loaded 5000 transcriptions


In [3]:
def plot_attention(envelope, time_env, attn_r, attn_v, attn_c,
                   v_intervals, c_intervals, word_boundaries,
                   text, utt_id, pred_r, pred_d, gt=None):
    """Plot envelope + all attention weights on a 4-row figure."""
    fig, axes = plt.subplots(4, 1, figsize=(18, 12), sharex=False,
                             gridspec_kw={'height_ratios': [2, 1, 1, 1]})
    fig.patch.set_facecolor('white')

    v_color = '#e74c3c'
    c_color = '#3498db'
    env_color = '#2c3e50'

    # --- Row 1: Envelope ---
    ax_env = axes[0]
    ax_env.plot(time_env, envelope, linewidth=0.6, color=env_color, alpha=0.9)

    # Shade V and C intervals
    for (start, end) in v_intervals:
        ax_env.axvspan(start, end, alpha=0.12, color=v_color)
    for (start, end) in c_intervals:
        ax_env.axvspan(start, end, alpha=0.08, color=c_color)

    # Word boundaries
    for word, w_start, w_end in word_boundaries:
        ax_env.axvline(w_start, color='gray', linestyle='--', linewidth=0.5, alpha=0.6)
        mid = (w_start + w_end) / 2
        ax_env.text(mid, ax_env.get_ylim()[1] * 0.95, word, ha='center', va='top',
                    fontsize=7, color='gray', alpha=0.8)

    ax_env.set_ylabel('Envelope', fontsize=10)
    title = f'{utt_id}: "{text}"'
    if gt is not None:
        title += f'  |  GT: {gt:.1f}'
    if pred_r is not None:
        title += f'  |  Rhythm pred: {pred_r:.2f}'
    if pred_d is not None:
        title += f'  |  Duration pred: {pred_d:.2f}'
    ax_env.set_title(title, fontsize=12, fontweight='bold')
    ax_env.set_xlim(0, time_env[-1])

    # --- Row 2: Rhythm attention ---
    ax_ra = axes[1]
    if attn_r is not None:
        n = len(attn_r)
        x_ra = np.linspace(0, time_env[-1], n)
        ax_ra.bar(x_ra, attn_r, width=x_ra[1]-x_ra[0] if n > 1 else 0.01,
                  color='#9b59b6', alpha=0.7, edgecolor='none')
        ax_ra.set_ylabel('Rhythm Attn', fontsize=10, color='#9b59b6')
    else:
        ax_ra.text(0.5, 0.5, 'No rhythm attention', ha='center', va='center', transform=ax_ra.transAxes)
    ax_ra.set_xlim(0, time_env[-1])

    # --- Row 3: Duration V attention ---
    ax_dv = axes[2]
    if attn_v is not None and len(v_intervals) > 0:
        mids = [(s + e) / 2 for s, e in v_intervals]
        n_v = min(len(attn_v), len(mids))
        ax_dv.vlines(mids[:n_v], 0, attn_v[:n_v], colors=v_color, linewidths=1)
        ax_dv.scatter(mids[:n_v], attn_v[:n_v], color=v_color, s=20, zorder=3)
        ax_dv.set_ylabel('V Attn', fontsize=10, color=v_color)
        ax_dv.set_ylim(bottom=0)
    else:
        ax_dv.text(0.5, 0.5, 'No V attention', ha='center', va='center', transform=ax_dv.transAxes)
    ax_dv.set_xlim(0, time_env[-1])

    # --- Row 4: Duration C attention ---
    ax_dc = axes[3]
    if attn_c is not None and len(c_intervals) > 0:
        mids = [(s + e) / 2 for s, e in c_intervals]
        n_c = min(len(attn_c), len(mids))
        ax_dc.vlines(mids[:n_c], 0, attn_c[:n_c], colors=c_color, linewidths=1)
        ax_dc.scatter(mids[:n_c], attn_c[:n_c], color=c_color, s=20, zorder=3)
        ax_dc.set_ylabel('C Attn', fontsize=10, color=c_color)
        ax_dc.set_ylim(bottom=0)
    else:
        ax_dc.text(0.5, 0.5, 'No C attention', ha='center', va='center', transform=ax_dc.transAxes)
    ax_dc.set_xlim(0, time_env[-1])
    ax_dc.set_xlabel('Time (s)', fontsize=10)

    plt.tight_layout()
    plt.show()

In [6]:
# --- Pick an utterance ---
UTT_ID = '000010011'
SPK_ID = 'SPEAKER0001'

key = resolve_key(UTT_ID, SPK_ID)
lookup_key = f'{SPK_ID}_{UTT_ID}'

# Load data from LMDB
data = load_utterance_from_lmdb(LMDB_PATH, key, items=['envelope_derivative', 'waveform'])
envelope = data['envelope_derivative']
waveform = data['waveform']
time_env = compute_envelope_time(len(envelope), len(waveform))

# Transcription
text = transcriptions.get(UTT_ID, '(no transcription)')

# Word boundaries
word_boundaries = load_word_boundaries(SPK_ID, UTT_ID)

# Ground truth score
label_col = duration_config['dataset_params'].get('label_column', 'fluency')
gt = load_utterance_labels(METADATA_PATH, key, label_col)

# Display transcription + audio
display(Markdown(f"**{UTT_ID}** — _{text}_  (GT: {gt})"))
display(Audio(waveform, rate=16000))

# --- Rhythm model ---
with torch.no_grad():
    batch_r = {'envelope': torch.tensor(envelope, dtype=torch.float32).unsqueeze(0).to(DEVICE)}
    pred_r, attn_r = rhythm_model(batch_r, return_attention=True)
    pred_r = pred_r.cpu().item()
    attn_r = attn_r.cpu().squeeze(0).numpy()

# --- Duration model ---
vc = load_vc_data(VC_FEATURES_PATH, lookup_key)
if vc:
    batch_d = build_vc_batch(vc, DEVICE)
    with torch.no_grad():
        pred_d, attn_d = duration_model(batch_d, return_attention=True)
        pred_d = pred_d.cpu().item()
        attn_v = attn_d['v'].cpu().squeeze(0).numpy()
        attn_c = attn_d['c'].cpu().squeeze(0).numpy()
else:
    pred_d, attn_v, attn_c = None, None, None

# V/C interval times for duration attention positioning
vc_intervals = load_vc_intervals(VC_ALIGNMENTS_PATH, lookup_key)

# --- Plot ---
plot_attention(
    envelope=envelope,
    time_env=time_env,
    attn_r=attn_r,
    attn_v=attn_v,
    attn_c=attn_c,
    v_intervals=vc_intervals.get('v_intervals', []),
    c_intervals=vc_intervals.get('c_intervals', []),
    word_boundaries=word_boundaries,
    text=text,
    utt_id=UTT_ID,
    pred_r=pred_r,
    pred_d=pred_d,
    gt=gt,
)

**000010011** — _WE CALL IT BEAR_  (GT: 9.0)

KeyError: 'envelope_derivative'